In [1]:
# ==========================================
# 1. SETUP & IMPORTATIONS
# ==========================================
import os, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Installation automatique de xgboost si absent
try:
    from xgboost import XGBClassifier
except Exception:
    import sys
    !{sys.executable} -m pip install xgboost
    from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ==========================================
# 2. EXERCISE 1 - EXPLORATORY DATA ANALYSIS
# ==========================================
# Extraction du dataset zip
ZIP_PATH = 'Heart Disease Prediction Dataset.zip'
EXTRACT_DIR = 'heart_ds'

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction réussie !")
else:
    print(f"Attention: {ZIP_PATH} introuvable. Assurez-vous d'avoir téléversé le fichier.")

# Liste des fichiers CSV
csv_files = glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True) or glob.glob(f"{EXTRACT_DIR}/*.csv")
print("Fichiers CSV trouvés :", csv_files)

csv_path = csv_files[0] if csv_files else None

# Chargement du DataFrame
if csv_path:
    df = pd.read_csv(csv_path)
else:
    print("Génération de données fictives pour éviter le blocage du code...")
    df = pd.DataFrame({
        'age': np.random.randint(29, 75, 200),
        'sex': np.random.choice(['M', 'F'], 200),
        'chol': np.random.randint(126, 564, 200),
        'target': np.random.choice([0, 1], 200)
    })

# Inspection rapide
print("\n--- Aperçu des données ---")
print(df.head())
print("\n--- Informations Générales ---")
print(df.info())

# Identification de la cible (Ajuster 'target' si le nom diffère dans votre fichier)
target = 'target' if 'target' in df.columns else df.columns[-1]

# Séparation Features / Target
X = df.drop(columns=[target])
y = df[target]

# Encodage de la cible si textuelle (ex: Yes/No)
if y.dtype == 'object':
    y = y.apply(lambda x: 1 if str(x).lower() in ['yes', '1', 'true'] else 0)

# Train Test Split avec stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f"\nForme finale - X_train: {X_train.shape}, X_test: {X_test.shape}")

# Visualisations de base
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cols_to_plot = numeric_features[:3]

if cols_to_plot:
    X_train[cols_to_plot].hist(bins=15, figsize=(10, 4))
    plt.tight_layout()
    plt.show()

y_train.value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title("Équilibre des classes (Target) - Train")
plt.xlabel("Classe")
plt.ylabel("Effectif")
plt.xticks(rotation=0)
plt.show()


# ==========================================
# 3. PREPROCESSING PIPELINE
# ==========================================
cat_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

pre = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)


# ==========================================
# 4. HELPER - EVALUATION FUNCTION
# ==========================================
def eval_and_report(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
    }
    
    print(f"\n=== Métriques pour {name} ===")
    for k, v in metrics.items():
        print(f"{k.capitalize()}: {v:.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax[0], cmap='Blues')
    ax[0].set_title(f"Confusion - {name}")

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc_score = roc_auc_score(y_test, y_proba)
        
        ax[1].plot(fpr, tpr, label=f"AUC = {auc_score:.4f}", color='darkorange', lw=2)
        ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        ax[1].set_xlabel("FPR")
        ax[1].set_ylabel("TPR")
        ax[1].set_title(f"ROC - {name}")
        ax[1].legend(loc="lower right")
    else:
        ax[1].text(0.5, 0.5, "predict_proba non disponible", ha='center', va='center')
        ax[1].set_title(f"ROC indisponible - {name}")
        
    plt.tight_layout()
    plt.show()

    return metrics


# ==========================================
# 5. ENTRAÎNEMENT ET ÉVALUATION DES MODÈLES
# ==========================================
summary = {}

# --- Exercice 2 : Logistic Regression (Sans GridSearch) ---
pipe_lr = Pipeline(steps=[('preprocessor', pre), ('lr', LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE))])
pipe_lr.fit(X_train, y_train)
summary['LR no grid'] = eval_and_report('LR no grid', pipe_lr, X_test, y_test)

# --- Exercice 3 : Logistic Regression (Avec GridSearch) ---
pipe_lr_cv = Pipeline(steps=[('preprocessor', pre), ('lr', LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE))])
param_grid_lr = {'lr__C': [0.01, 0.1, 1, 10, 100], 'lr__penalty': ['l1', 'l2']}
grid_lr = GridSearchCV(pipe_lr_cv, param_grid_lr, cv=5, scoring='f1', n_jobs=-1)
grid_lr.fit(X_train, y_train)
summary['LR grid'] = eval_and_report('LR grid', grid_lr.best_estimator_, X_test, y_test)

# --- Exercice 4 : SVM (Sans GridSearch) ---
pipe_svm = Pipeline(steps=[('preprocessor', pre), ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE))])
pipe_svm.fit(X_train, y_train)
summary['SVM no grid'] = eval_and_report('SVM no grid', pipe_svm, X_test, y_test)

# --- Exercice 5 : SVM (Avec GridSearch) ---
pipe_svm_cv = Pipeline(steps=[('preprocessor', pre), ('svm', SVC(probability=True, random_state=RANDOM_STATE))])
param_grid_svm = {'svm__kernel': ['rbf', 'linear'], 'svm__C': [0.1, 1, 10], 'svm__gamma': ['scale', 'auto', 0.1]}
grid_svm = GridSearchCV(pipe_svm_cv, param_grid_svm, cv=5, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train, y_train)
summary['SVM grid'] = eval_and_report('SVM grid', grid_svm.best_estimator_, X_test, y_test)

# --- Exercice 6 : XGBoost (Sans GridSearch) ---
pipe_xgb = Pipeline(steps=[('preprocessor', pre), ('xgb', XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE))])
pipe_xgb.fit(X_train, y_train)
summary['XGB no grid'] = eval_and_report('XGB no grid', pipe_xgb, X_test, y_test)

# --- Exercice 7 : XGBoost (Avec GridSearch) ---
pipe_xgb_cv = Pipeline(steps=[('preprocessor', pre), ('xgb', XGBClassifier(random_state=RANDOM_STATE))])
param_grid_xgb = {'xgb__n_estimators': [100, 200], 'xgb__learning_rate': [0.05, 0.1], 'xgb__max_depth': [3, 4]}
grid_xgb = GridSearchCV(pipe_xgb_cv, param_grid_xgb, cv=5, scoring='f1', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
summary['XGB grid'] = eval_and_report('XGB grid', grid_xgb.best_estimator_, X_test, y_test)


# ==========================================
# 6. COMPARAISON FINALE
# ==========================================
df_compare = pd.DataFrame.from_dict(summary, orient='index')
print("\n=== TABLEAU COMPARATIF DES PERFORMANCES ===")
print(df_compare.round(4))

'c:\Users\yaol3\Desktop\SIRA' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


ModuleNotFoundError: No module named 'xgboost'